## MANC mirror landmarks
In this notebook, we will generate a "shortcut" to mirror MANC data. The general idea is:
1. Evenly sample points from one side of the brain
2. Flip them to the other side to generate mirror-symmetrical source landmarks
3. Mirror those landmarks again but going via JRCVNC2018M to generate the target landmarks

In [ ]:
import navis
import flybrains

import numpy as np
import pandas as pd

In [ ]:
np.array(flybrains.MANC.boundingbox).reshape(3, 2)

In [ ]:
center = 49184 + (342752 - 49184) / 2
center

In [ ]:
# Sample points within the volume
offset = 20_000
res = 20_000
sample = np.mgrid[49184-offset:342752+offset:res, 40984-offset:406872+offset:res, max(0, 16208-offset):518552+offset:res].reshape(3,-1).T

# The MANC volume is tilted so many of these axis-aligned sample points
# will be far outside the actual volume.
import ncollpyde
vol = ncollpyde.Volume(flybrains.MANC.mesh.vertices, flybrains.MANC.mesh.faces)

# Distance to surface
d = vol.distance(sample)

# Drop everything farther than 20um outside the surface
sample = sample[d < 10_000]

sample.shape

In [ ]:
# The properly mirrored coordinates
sample_mirr = navis.mirror_brain(sample, template='MANC', via='JRCVNC2018M', verbose=True)

In [ ]:
# Just flipped
sample_flip = navis.mirror_brain(sample, template='MANC', warp=False)

In [ ]:
# Just flipped
navis.plot3d([flybrains.MANC, sample_flip])

In [ ]:
# Properly mirrored
navis.plot3d([flybrains.MANC, sample_mirr])

In [ ]:
# Bring it together
source = pd.DataFrame(sample_flip, columns=['x_flip', 'y_flip', 'z_flip']).round().astype(int)
target = pd.DataFrame(sample_mirr, columns=['x_mirr', 'y_mirr', 'z_mirr']).round().astype(int)
lm = pd.concat((source, target), axis=1)
lm.head()

In [ ]:
lm.to_csv('MANC_mirror_landmarks.csv', index=False)